In [ ]:
# ============================================================
# MIMICEL -> PALSYN Input Converter
# Input : ./data/mimicel_train.csv / mimicel_val.csv / mimicel_test.csv
# Output: ./results/palsyn_input
# ============================================================

from pathlib import Path
import json
import pandas as pd
import numpy as np


# ------------------------------------------------------------
# 0. Config
# ------------------------------------------------------------

DATA_DIR = Path("./../../MIMICEL_data/")
OUT_DIR = Path("./results/")
OUT_DIR.mkdir(parents=True, exist_ok=True)

CASE_COL = "stay_id"
TIME_COL = "timestamps"
ACT_COL = "activity"

SPLITS = {
    "train": DATA_DIR / "mimicel_train.csv",
    "val": DATA_DIR / "mimicel_val.csv",
    "test": DATA_DIR / "mimicel_test.csv",
}

# PALSYN에 포함할 multi-perspective columns
CASE_ATTR_COLS = [
    "gender",
    "race",
    "arrival_transport",
    "disposition",
    "acuity",
    "chiefcomplaint",
]

EVENT_ATTR_COLS = [
    "temperature",
    "heartrate",
    "resprate",
    "o2sat",
    "sbp",
    "dbp",
    "pain",
    "rhythm",
    "icd_code",
    "icd_version",
    "icd_title",
    "name",
    "gsn",
    "ndc",
    "etccode",
    "etcdescription",
]

NUMERIC_COLS = [
    "temperature",
    "heartrate",
    "resprate",
    "o2sat",
    "sbp",
    "dbp",
    "pain",
    "acuity",
]

CATEGORICAL_COLS = [
    "gender",
    "race",
    "arrival_transport",
    "disposition",
    "chiefcomplaint",
    "rhythm",
    "icd_code",
    "icd_version",
    "icd_title",
    "name",
    "gsn",
    "ndc",
    "etccode",
    "etcdescription",
]


# ------------------------------------------------------------
# 1. Load splits
# ------------------------------------------------------------

dfs = {}

for split, path in SPLITS.items():
    if not path.exists():
        print(f"[SKIP] {split}: file not found -> {path}")
        continue

    print(f"[LOAD] {split}: {path}")
    df = pd.read_csv(path, low_memory=False)
    df.columns = [c.strip() for c in df.columns]

    if TIME_COL not in df.columns:
        if "timestamp" in df.columns:
            TIME_COL = "timestamp"
        else:
            raise ValueError("timestamp 컬럼 없음")

    required_cols = [CASE_COL, TIME_COL, ACT_COL]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"필수 컬럼 없음: {col}")

    df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors="coerce")
    df = df.dropna(subset=[CASE_COL, TIME_COL, ACT_COL])
    df = df.sort_values([CASE_COL, TIME_COL]).reset_index(drop=True)

    dfs[split] = df

if "train" not in dfs:
    raise FileNotFoundError("PALSYN 학습에는 ./data/mimicel_train.csv가 필요합니다.")


# ------------------------------------------------------------
# 2. Select available columns
# ------------------------------------------------------------

all_cols = set(dfs["train"].columns)

case_attr_cols = [c for c in CASE_ATTR_COLS if c in all_cols]
event_attr_cols = [c for c in EVENT_ATTR_COLS if c in all_cols]
numeric_cols = [c for c in NUMERIC_COLS if c in all_cols]
categorical_cols = [c for c in CATEGORICAL_COLS if c in all_cols]

print("[COLUMNS]")
print("case_attr_cols:", case_attr_cols)
print("event_attr_cols:", event_attr_cols)
print("numeric_cols:", numeric_cols)
print("categorical_cols:", categorical_cols)


# ------------------------------------------------------------
# 3. Numeric binning based on TRAIN only
# ------------------------------------------------------------

print("[BINNING] Build numeric bins from train only")

numeric_bin_meta = {}

N_BINS = 10

for col in numeric_cols:
    s = pd.to_numeric(dfs["train"][col], errors="coerce").dropna()

    if len(s) == 0:
        continue

    quantiles = np.linspace(0, 1, N_BINS + 1)
    bins = s.quantile(quantiles).drop_duplicates().values.tolist()

    if len(bins) < 2:
        bins = [float(s.min()), float(s.max()) + 1.0]

    numeric_bin_meta[col] = bins

with open(OUT_DIR / "numeric_bin_meta.json", "w", encoding="utf-8") as f:
    json.dump(numeric_bin_meta, f, ensure_ascii=False, indent=2)

print("numeric bins saved")


def numeric_to_bin_token(col, value):
    if col not in numeric_bin_meta:
        return None

    try:
        x = float(value)
    except Exception:
        return None

    if pd.isna(x):
        return None

    bins = numeric_bin_meta[col]

    for i in range(len(bins) - 1):
        if bins[i] <= x <= bins[i + 1]:
            return f"{col}=BIN_{i}"

    if x < bins[0]:
        return f"{col}=BIN_0"

    return f"{col}=BIN_{len(bins) - 2}"


# ------------------------------------------------------------
# 4. Time tokenization
# ------------------------------------------------------------

def make_time_tokens(times):
    tokens = []

    if len(times) == 0:
        return tokens

    start = times[0]
    prev = times[0]

    for t in times:
        delta_prev = max((t - prev).total_seconds(), 0.0)
        delta_start = max((t - start).total_seconds(), 0.0)

        if delta_prev == 0:
            delta_token = "delta=0"
        elif delta_prev <= 60:
            delta_token = "delta=0_1min"
        elif delta_prev <= 300:
            delta_token = "delta=1_5min"
        elif delta_prev <= 1800:
            delta_token = "delta=5_30min"
        elif delta_prev <= 3600:
            delta_token = "delta=30_60min"
        elif delta_prev <= 21600:
            delta_token = "delta=1_6h"
        else:
            delta_token = "delta=6h_plus"

        if delta_start <= 3600:
            elapsed_token = "elapsed=0_1h"
        elif delta_start <= 21600:
            elapsed_token = "elapsed=1_6h"
        elif delta_start <= 43200:
            elapsed_token = "elapsed=6_12h"
        else:
            elapsed_token = "elapsed=12h_plus"

        tokens.append((delta_token, elapsed_token))
        prev = t

    return tokens


# ------------------------------------------------------------
# 5. PALSYN sequence converter
# ------------------------------------------------------------

def clean_token_value(x):
    if pd.isna(x):
        return None

    x = str(x).strip()

    if x == "" or x.lower() in ["nan", "none", "null"]:
        return None

    x = x.replace(" ", "_")
    x = x.replace("/", "_")
    x = x.replace(",", "_")
    x = x.replace(";", "_")
    x = x.replace("|", "_")

    return x


def convert_split_to_palsyn(df, split_name):
    """
    Output:
    - {split}_palsyn_sequences.txt
    - {split}_palsyn_cases.jsonl
    """

    sequence_lines = []
    case_records = []

    skipped_cases = 0

    for case_id, g in df.groupby(CASE_COL, sort=False):
        g = g.sort_values(TIME_COL)

        if len(g) < 2:
            skipped_cases += 1
            continue

        case_tokens = []
        event_tokens = []

        # case-level attributes: 첫 event 기준 사용
        first_row = g.iloc[0]

        for col in case_attr_cols:
            if col in numeric_cols:
                token = numeric_to_bin_token(col, first_row.get(col))
                if token:
                    case_tokens.append(f"CASE_{token}")
            else:
                value = clean_token_value(first_row.get(col))
                if value:
                    case_tokens.append(f"CASE_{col}={value}")

        times = g[TIME_COL].tolist()
        time_tokens = make_time_tokens(times)

        for idx, (_, row) in enumerate(g.iterrows()):
            current_tokens = []

            # activity token
            act_value = clean_token_value(row[ACT_COL])
            if act_value is None:
                continue

            current_tokens.append(f"ACT={act_value}")

            # time tokens
            delta_token, elapsed_token = time_tokens[idx]
            current_tokens.append(delta_token)
            current_tokens.append(elapsed_token)

            # event-level attributes
            for col in event_attr_cols:
                if col in numeric_cols:
                    token = numeric_to_bin_token(col, row.get(col))
                    if token:
                        current_tokens.append(f"EVT_{token}")
                else:
                    value = clean_token_value(row.get(col))
                    if value:
                        current_tokens.append(f"EVT_{col}={value}")

            # event boundary
            event_tokens.extend(current_tokens)
            event_tokens.append("<EVT_END>")

        if len(event_tokens) == 0:
            skipped_cases += 1
            continue

        full_sequence = ["<CASE_START>"] + case_tokens + ["<TRACE_START>"] + event_tokens + ["<TRACE_END>"]

        sequence_lines.append(" ".join(full_sequence))

        case_records.append({
            "case_id": str(case_id),
            "num_events": int(len(g)),
            "num_tokens": int(len(full_sequence)),
            "sequence": full_sequence,
        })

    seq_path = OUT_DIR / f"{split_name}_palsyn_sequences.txt"
    jsonl_path = OUT_DIR / f"{split_name}_palsyn_cases.jsonl"

    seq_path.write_text("\n".join(sequence_lines), encoding="utf-8")

    with open(jsonl_path, "w", encoding="utf-8") as f:
        for rec in case_records:
            f.write(json.dumps(rec, ensure_ascii=False) + "\n")

    summary = {
        "split": split_name,
        "num_cases_written": len(sequence_lines),
        "skipped_cases": skipped_cases,
        "seq_file": str(seq_path),
        "jsonl_file": str(jsonl_path),
    }

    return summary


# ------------------------------------------------------------
# 6. Convert train / val / test
# ------------------------------------------------------------

summaries = []

for split_name, df in dfs.items():
    print(f"[CONVERT] {split_name}")
    summary = convert_split_to_palsyn(df, split_name)
    summaries.append(summary)
    print(summary)


# ------------------------------------------------------------
# 7. Build vocabulary from TRAIN only
# ------------------------------------------------------------

print("[VOCAB] Build PALSYN vocabulary from train only")

train_seq_path = OUT_DIR / "train_palsyn_sequences.txt"

vocab = {
    "<PAD>": 0,
    "<UNK>": 1,
}

with open(train_seq_path, "r", encoding="utf-8") as f:
    for line in f:
        for tok in line.strip().split():
            if tok not in vocab:
                vocab[tok] = len(vocab)

id2token = {idx: tok for tok, idx in vocab.items()}

with open(OUT_DIR / "token2id.json", "w", encoding="utf-8") as f:
    json.dump(vocab, f, ensure_ascii=False, indent=2)

with open(OUT_DIR / "id2token.json", "w", encoding="utf-8") as f:
    json.dump(id2token, f, ensure_ascii=False, indent=2)

print("vocab size:", len(vocab))


# ------------------------------------------------------------
# 8. Encode sequences to id format
# ------------------------------------------------------------

def encode_sequence_file(split_name):
    src_path = OUT_DIR / f"{split_name}_palsyn_sequences.txt"
    dst_path = OUT_DIR / f"{split_name}_palsyn_token_ids.txt"

    lines = []

    with open(src_path, "r", encoding="utf-8") as f:
        for line in f:
            ids = [str(vocab.get(tok, vocab["<UNK>"])) for tok in line.strip().split()]
            lines.append(" ".join(ids))

    dst_path.write_text("\n".join(lines), encoding="utf-8")

    return str(dst_path)


encoded_paths = {}

for split_name in dfs.keys():
    encoded_paths[split_name] = encode_sequence_file(split_name)

print(encoded_paths)


# ------------------------------------------------------------
# 9. Save summary
# ------------------------------------------------------------

summary_df = pd.DataFrame(summaries)
summary_df.to_csv(OUT_DIR / "palsyn_encoding_summary.csv", index=False)

config = {
    "case_col": CASE_COL,
    "time_col": TIME_COL,
    "activity_col": ACT_COL,
    "case_attr_cols": case_attr_cols,
    "event_attr_cols": event_attr_cols,
    "numeric_cols": numeric_cols,
    "categorical_cols": categorical_cols,
    "output_dir": str(OUT_DIR),
    "vocab_size": len(vocab),
    "encoded_paths": encoded_paths,
}

with open(OUT_DIR / "palsyn_preprocess_config.json", "w", encoding="utf-8") as f:
    json.dump(config, f, ensure_ascii=False, indent=2)

print("\nDone.")
print(f"Outputs saved to: {OUT_DIR.resolve()}")

[LOAD] train: ..\..\MIMICEL_data\mimicel_train.csv
[LOAD] val: ..\..\MIMICEL_data\mimicel_val.csv
[LOAD] test: ..\..\MIMICEL_data\mimicel_test.csv
[COLUMNS]
case_attr_cols: ['gender', 'race', 'arrival_transport', 'disposition', 'acuity', 'chiefcomplaint']
event_attr_cols: ['temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp', 'pain', 'rhythm', 'icd_code', 'icd_version', 'icd_title', 'name', 'gsn', 'ndc', 'etccode', 'etcdescription']
numeric_cols: ['temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp', 'pain', 'acuity']
categorical_cols: ['gender', 'race', 'arrival_transport', 'disposition', 'chiefcomplaint', 'rhythm', 'icd_code', 'icd_version', 'icd_title', 'name', 'gsn', 'ndc', 'etccode', 'etcdescription']
[BINNING] Build numeric bins from train only
numeric bins saved
[CONVERT] train
{'split': 'train', 'num_cases_written': 297519, 'skipped_cases': 0, 'seq_file': 'results\\palsyn_input\\train_palsyn_sequences.txt', 'jsonl_file': 'results\\palsyn_input\\train_palsyn_ca